In [2]:
import os
import tifffile as tiff

# Directories
oct_volume_dir = "/media/admin/Expansion/Mosaic_Data_for_Ipeks_Group/OCT_Volumes/"
oct_cutoff_dir = "/media/admin/Expansion/Mosaic_Data_for_Ipeks_Group/OCT_VolumeSplit_Indices/"

# Scan files
vol_files = [f for f in os.listdir(oct_volume_dir) if f.endswith("_OCT_uint16.tif")]

print(f"{'Volume':<25} {'#B-scans':>8} {'1st Cutoff':>12} {'Last Cutoff':>14} {'OK?':>6}")
print("-" * 70)

for vol_file in sorted(vol_files):
    base = vol_file.replace("_OCT_uint16.tif", "")
    cutoff_file = f"{base}_OCT_Cutoff_Indices.txt"
    cutoff_path = os.path.join(oct_cutoff_dir, cutoff_file)

    if not os.path.exists(cutoff_path):
        continue

    with tiff.TiffFile(os.path.join(oct_volume_dir, vol_file)) as tif:
        total_slices = len(tif.pages)

    with open(cutoff_path, "r") as f:
        cutoffs = [int(line.strip()) for line in f if line.strip().isdigit()]

    if not cutoffs:
        continue

    first_cutoff = cutoffs[0]
    last_cutoff = cutoffs[-1]

    ok_first = first_cutoff <= 4000
    ok_last = (total_slices - last_cutoff) <= 4000
    is_ok = ok_first and ok_last

    if not is_ok:
        print(f"{base:<25} {total_slices:>8} {first_cutoff:>12} {last_cutoff:>14} {'NO':>6}")


Volume                    #B-scans   1st Cutoff    Last Cutoff    OK?
----------------------------------------------------------------------
1.2                          12407          219           8219     NO
1.4                          10378         1778           5778     NO
15.1                          9614          765           4765     NO
19.2_v2                      14339         1674           9674     NO
19.3                         17885         1153          13153     NO
23.2                         13626         4759          12759     NO
24.1_v1                       9615         1080           5080     NO
24.4                         15231         2707          10707     NO
25.2                         12316         3945           7945     NO
27.2_v4                      12750         4369           8369     NO
31.3                         12967          384           8384     NO
32.3                         16584         3893          11893     NO
33.4               

In [3]:
# Scan cutoff files
cutoff_files = [f for f in os.listdir(oct_cutoff_dir) if f.endswith("_OCT_Cutoff_Indices.txt")]

print(f"{'File':<35} {'Cutoffs'}")
print("-" * 60)

for filename in sorted(cutoff_files):
    path = os.path.join(oct_cutoff_dir, filename)

    with open(path, "r") as f:
        cutoffs = [int(line.strip()) for line in f if line.strip().isdigit()]

    diffs = [b - a for a, b in zip(cutoffs[:-1], cutoffs[1:])]
    if not all(diff == 4000 for diff in diffs):
        print(f"{filename:<35} {cutoffs}")

File                                Cutoffs
------------------------------------------------------------
14.4_OCT_Cutoff_Indices.txt         [1946, 5046, 9046]
